In [ ]:
from io import BytesIO

audio_file = open("my_recording.wav", "rb")
buffer = BytesIO()

In [ ]:
with open("my_recording.wav", "rb") as wf:
    open(buffer, "wb").write(wf)

In [ ]:
from openai import OpenAI

def speech_to_text(audio_file):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    transcription = client.audio.transcriptions.create(
        model="gpt-4o-transcribe", 
        file=audio_file
    )
    return transcription.text

print(speech_to_text(audio_file))

In [ ]:
import pyaudio
import wave
import numpy as np

from io import BytesIO
import speech_recognition as sr
from openai import OpenAI

def record_on_voice_stop(silence_threshold=500, silence_duration=2, chunk=1024, rate=44100):
    p = pyaudio.PyAudio()

    stream = p.open(format=pyaudio.paInt16,
                    channels=1,
                    rate=rate,
                    input=True,
                    frames_per_buffer=chunk)

    print("Listening... Start talking.")

    frames = []
    silent_chunks = 0
    max_silent_chunks = int(rate / chunk * silence_duration)

    while True:
        data = stream.read(chunk)
        frames.append(data)

        audio_data = np.frombuffer(data, dtype=np.int16)
        volume = np.abs(audio_data).mean()

        if volume < silence_threshold:
            silent_chunks += 1
        else:
            silent_chunks = 0

        if silent_chunks > max_silent_chunks:
            break

    stream.stop_stream()
    stream.close()
    p.terminate()
    buffer = BytesIO()
    with wave.open(buffer, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(p.get_sample_size(pyaudio.paInt16))
        wf.setframerate(rate)
        wf.writeframes(b''.join(frames))
    
    buffer.seek(0)
    buffer.name = "audio_file.wav"
    return buffer

In [ ]:
buffer = record_on_voice_stop()

In [ ]:
audio_file

In [ ]:
buffer

In [ ]:
from openai import OpenAI

def speech_to_text(audio_file):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    transcription = client.audio.transcriptions.create(
        model="gpt-4o-transcribe", 
        file=audio_file
    )
    return transcription.text

print(speech_to_text(buffer))

In [ ]:
import whisper_sourced.whisper as wh

model = wh.load_model("turbo")
result = model.transcribe("my_recording.wav")

In [ ]:
result

In [ ]:
import requests
import json

# messages:list = [
#    {"role":"user/assisant", "content":"some text"}
# ]

def ai_res(messages:list) -> None:
    session = requests.Session()
    session.auth = ("cookie", "test12345!")

    data:dict = {
        "model":"llama3.1",
        "messages":messages
    }

    res:requests.models.Response = session.post("https://api.zuzu.im/api/chat", json=data, stream=True)
    for line in res.iter_lines():
        print(json.loads(line)["message"]["role"], end=" ")

ai_res([{"role":"user", "content":"what is going on?"}])

In [ ]:
import openai

openai.api_key = os.environ["OPENAI_API_KEY"]

def solve_math_from_image():
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages=messages,
        max_tokens=1000,
    )
    return response.choices[0].message.content

print(solve_math_from_image())